# Phase 15 — Stage 2: DPO Training with LoRA Adapter
## CodeMentor-LLM
Aligning fine-tuned Llama-3.2-3B-Instruct using
Direct Preference Optimization (DPO).

## Goal
- Load SFT checkpoint as base
- Apply fresh LoRA adapter
- Train on preference dataset (chosen/rejected pairs)
- Push DPO model to HuggingFace Hub

## Config
- Base     : Abdulmoiz123/codementor-llm-sft
- Dataset  : Abdulmoiz123/codementor-llm-preference
- Beta     : 0.1
- Epochs   : 1
- LR       : 5e-5

# libraries

In [ ]:
!pip uninstall numpy -y
!pip install numpy==1.26.4
!pip install -q transformers==4.49.0 peft==0.14.0 trl==0.15.2 bitsandbytes==0.45.3 accelerate==1.5.1 datasets==3.3.2 wandb

# Login to HuggingFace and W&B

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import wandb

# Login to HuggingFace
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print("HuggingFace login successful")

# Login to W&B
wandb_api_key = userdata.get('WANDB_API_KEY')
wandb.login(key=wandb_api_key)
print("W&B login successful")

# Load SFT model and tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel, LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
import torch

model_id = "meta-llama/Llama-3.2-3B-Instruct"
sft_adapter_id = "Abdulmoiz123/codementor-llm-sft"

# Quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Load base model
print("Loading base model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)

# Load SFT adapter
print("Loading SFT adapter...")
model = PeftModel.from_pretrained(
    base_model,
    sft_adapter_id,
    is_trainable=True,
)

print(f"Model loaded — {model.get_memory_footprint() / 1024**3:.2f} GB")

#  Load preference dataset

In [ ]:
from datasets import load_dataset

# Load preference dataset
print("Loading preference dataset...")
preference_dataset = load_dataset("Abdulmoiz123/codementor-llm-preference")
print(f"Dataset loaded: {preference_dataset}")
print(f"\nSample:")
print(f"Prompt  : {preference_dataset['train'][0]['prompt'][:100]}")
print(f"Chosen  : {preference_dataset['train'][0]['chosen'][:100]}")
print(f"Rejected: {preference_dataset['train'][0]['rejected'][:100]}")

#  Initialize W&B run

In [ ]:
wandb.init(
    project="codementor-llm",
    name="dpo-llama3-3b",
    config={
        "model": "meta-llama/Llama-3.2-3B-Instruct",
        "sft_adapter": "Abdulmoiz123/codementor-llm-sft",
        "dataset": "Abdulmoiz123/codementor-llm-preference",
        "beta": 0.1,
        "epochs": 1,
        "learning_rate": 5e-5,
        "batch_size": 4,
    }
)

print("W&B run initialized successfully")
print(f"Run URL: {wandb.run.url}")

# Configure and run DPO Training

In [ ]:
from trl import DPOTrainer, DPOConfig

# DPO Config
dpo_config = DPOConfig(
    output_dir="./checkpoints/dpo",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="wandb",
    push_to_hub=True,
    hub_model_id="Abdulmoiz123/codementor-llm-dpo",
    hub_strategy="every_save",
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    beta=0.1,
    max_length=512,
    max_prompt_length=256,
)

# Initialize DPO Trainer
trainer = DPOTrainer(
    model=model,
    args=dpo_config,
    train_dataset=preference_dataset["train"],
    processing_class=tokenizer,
)

print("DPOTrainer initialized successfully")
print(f"Training samples: {len(preference_dataset['train'])}")

#  Start DPO Training

In [ ]:
# Start training
trainer.train()

print("\nDPO Training complete!")

# Push final model

In [ ]:
# Push final model to HF Hub
trainer.push_to_hub()
print("DPO model pushed to HuggingFace Hub successfully")
print(f"Model: https://huggingface.co/Abdulmoiz123/codementor-llm-dpo")

# Finish W&B
wandb.finish()
print("W&B run finished")